
## Usage Instructions

To run this pipeline:

1. **Update data paths** in Cell 5 to match your file locations
2. **Run cells sequentially** - each cell depends on previous ones
3. **Monitor progress** - training progress is displayed for each step
4. **Adjust parameters** if needed:
   - VAE architecture in cells 17, 24, 27
   - Training epochs and learning rates in trainer calls
   - Feature filtering parameters in Cell 8
   - Sparse autoencoder parameters in SAE training calls

The pipeline is designed to handle various data quality issues and will provide warnings when problems are detected. Results and trained models are automatically saved for future use.# Microbiome IBD Classification Pipeline with VAE+SAE Feature Engineering
*Adapted for metagenomics and metatranscriptomics data*

This notebook implements a comprehensive pipeline for IBD classification using Variational Autoencoders (VAE) and Sparse Autoencoders (SAE) for feature engineering on microbiome data.




## Cell 1: Imports and Setup


In [2]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import umap
import biom
from qiime2 import Metadata
from gemelli.preprocessing import matrix_rclr
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


All libraries imported successfully!
PyTorch version: 1.12.1
CUDA available: False



## Cell 2: Helper Functions


In [3]:
def get_device():
    """Get the appropriate device for computation"""
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS (Metal Performance Shaders) device")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using CUDA device")
    else:
        device = torch.device("cpu")
        print("Using CPU device")
    return device

def add_noise(data, noise_factor=0.05):
    """Add noise to input data for regularization"""
    noise = torch.randn_like(data) * noise_factor * data.std()
    return data + noise

def plot_training_curves(train_losses, val_losses=None, title="Training Curves"):
    """Plot training and validation loss curves"""
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss', alpha=0.7)
    if val_losses is not None and len(val_losses) > 0:
        plt.plot(val_losses, label='Validation Loss', alpha=0.7)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Initialize device
device = get_device()
print(f"Device initialized: {device}")


Using MPS (Metal Performance Shaders) device
Device initialized: mps


## Cell 3: Taxonomy Helper Functions


In [4]:
def find_ogu_by_species(taxonomy_df, species_name, feature_data):
    """Find OGU IDs corresponding to a given species name"""
    matching_ogus = []
    for ogu_id in taxonomy_df.index:
        if ogu_id in feature_data.columns:
            taxon = taxonomy_df.loc[ogu_id, 'taxonomy']
            if species_name.lower() in taxon.lower():
                matching_ogus.append(ogu_id)
    return matching_ogus

def get_taxonomy_info(ogu_id, taxonomy_df):
    """Get full taxonomy information for an OGU ID"""
    if ogu_id in taxonomy_df.index:
        return taxonomy_df.loc[ogu_id, 'taxonomy']
    return "Unknown taxonomy"

def extract_species_name(taxonomy_string):
    """Extract species name from taxonomy string"""
    if 's__' in taxonomy_string:
        # Get the species part after 's__'
        species_part = taxonomy_string.split('s__')[-1]
        # Remove any trailing taxonomic levels and clean up
        species = species_part.split(';')[0].strip()
        if species and species != '' and species != 'unclassified' and species != 'unidentified':
            return species
    
    # If no species, try genus
    if 'g__' in taxonomy_string:
        genus_part = taxonomy_string.split('g__')[-1]
        genus = genus_part.split(';')[0].strip()
        if genus and genus != '' and genus != 'unclassified' and genus != 'unidentified':
            return f"genus_{genus}"
    
    # If no genus, try family
    if 'f__' in taxonomy_string:
        family_part = taxonomy_string.split('f__')[-1]
        family = family_part.split(';')[0].strip()
        if family and family != '' and family != 'unclassified' and family != 'unidentified':
            return f"family_{family}"
    
    return None

def format_feature_name(feature_id, taxonomy_df=None):
    """Format feature name to show both OGU ID and species/genus/family name"""
    if taxonomy_df is not None and feature_id in taxonomy_df.index:
        taxonomy = taxonomy_df.loc[feature_id, 'taxonomy']
        species_name = extract_species_name(taxonomy)
        if species_name:
            return f"{feature_id} ({species_name})"
    return feature_id

def get_taxonomic_level(taxonomy_string, level='species'):
    """Extract specific taxonomic level from taxonomy string"""
    level_prefixes = {
        'domain': 'd__',
        'phylum': 'p__',
        'class': 'c__', 
        'order': 'o__',
        'family': 'f__',
        'genus': 'g__',
        'species': 's__'
    }
    
    if level in level_prefixes:
        prefix = level_prefixes[level]
        if prefix in taxonomy_string:
            level_part = taxonomy_string.split(prefix)[-1]
            return level_part.split(';')[0].strip()
    
    return "unclassified"

print("Taxonomy helper functions defined successfully!")


Taxonomy helper functions defined successfully!



## Cell 4: Data Processor Class



In [5]:
class MicrobiomeDataProcessor:
    """Class to handle microbiome data loading and preprocessing"""
    
    def __init__(self, data_paths):
        self.data_paths = data_paths
        self.device = device
        
    def load_data(self):
        """Load microbiome data from biom files and metadata"""
        print("Loading microbiome data...")
        
        # Load biom tables
        self.table_metag = biom.load_table(self.data_paths['metag_biom'])
        self.table_metaT = biom.load_table(self.data_paths['metat_biom'])
        
        # Convert to dataframes
        self.table_metag_df = self.table_metag.to_dataframe().T
        self.table_metaT_df = self.table_metaT.to_dataframe().T
        
        # Load metadata
        self.metadata = Metadata.load(self.data_paths['metadata'])
        self.metadata_df = self.metadata.to_dataframe()
        
        # Load lineages if available
        if 'lineages' in self.data_paths:
            self.lineages = pd.read_csv(self.data_paths['lineages'], sep='\t', header=None)
            self.lineages.columns = ['genome_id', 'taxonomy']
            self.process_taxonomy()
        
        print(f"MetaG shape: {self.table_metag_df.shape}")
        print(f"MetaT shape: {self.table_metaT_df.shape}")
        print(f"Metadata shape: {self.metadata_df.shape}")
        print(f"Diagnosis distribution:\n{self.metadata_df['diagnosis'].value_counts()}")
        
    def process_taxonomy(self):
        """Process taxonomy information for features"""
        metag_features = list(self.table_metag_df.columns)
        metaT_features = list(self.table_metaT_df.columns)
        
        # Filter for features in the data
        self.metag_taxonomy = self.lineages[self.lineages['genome_id'].isin(metag_features)]
        self.metaT_taxonomy = self.lineages[self.lineages['genome_id'].isin(metaT_features)]
        
        print(f"MetaG features with taxonomy: {len(self.metag_taxonomy)}")
        print(f"MetaT features with taxonomy: {len(self.metaT_taxonomy)}")
        
    def apply_rclr_transformation(self, pseudocount=1e-6):
        """Apply robust CLR transformation using gemelli with pseudocount"""
        print(f"Applying RCLR transformation with pseudocount: {pseudocount}...")
        
        # Apply RCLR to metagenomics data
        table_metag_array = self.table_metag.matrix_data.toarray().T
        table_metag_array_pseudo = table_metag_array + pseudocount
        self.table_metag_rclr = pd.DataFrame(
            matrix_rclr(table_metag_array_pseudo),
            index=self.table_metag.ids('sample'),
            columns=self.table_metag.ids('observation')
        )
        
        # Apply RCLR to metatranscriptomics data
        table_metaT_array = self.table_metaT.matrix_data.toarray().T
        table_metaT_array_pseudo = table_metaT_array + pseudocount
        self.table_metaT_rclr = pd.DataFrame(
            matrix_rclr(table_metaT_array_pseudo),
            index=self.table_metaT.ids('sample'),
            columns=self.table_metaT.ids('observation')
        )
        
        print(f"MetaG RCLR shape: {self.table_metag_rclr.shape}")
        print(f"MetaT RCLR shape: {self.table_metaT_rclr.shape}")
        
    def filter_features(self, min_prevalence=0.1, top_variance_pct=0.20):
        """Filter features based on prevalence and variance"""
        print(f"Filtering features (min_prevalence={min_prevalence}, top_variance={top_variance_pct})...")
        
        # Filter MetaG features
        metag_prevalence = (self.table_metag_rclr != 0).mean(axis=0)
        metag_filtered = self.table_metag_rclr.loc[:, metag_prevalence >= min_prevalence]
        
        # Keep top variance features
        metag_variance = metag_filtered.var(axis=0)
        top_n = int(len(metag_variance) * top_variance_pct)
        metag_top_features = metag_variance.nlargest(top_n).index
        self.table_metag_filtered = metag_filtered[metag_top_features]
        
        # Filter MetaT features
        metaT_prevalence = (self.table_metaT_rclr != 0).mean(axis=0)
        metaT_filtered = self.table_metaT_rclr.loc[:, metaT_prevalence >= min_prevalence]
        
        # Keep top variance features
        metaT_variance = metaT_filtered.var(axis=0)
        top_n = int(len(metaT_variance) * top_variance_pct)
        metaT_top_features = metaT_variance.nlargest(top_n).index
        self.table_metaT_filtered = metaT_filtered[metaT_top_features]
        
        print(f"MetaG features after filtering: {self.table_metag_filtered.shape[1]}")
        print(f"MetaT features after filtering: {self.table_metaT_filtered.shape[1]}")
        
    def prepare_datasets(self):
        """Prepare datasets for training"""
        print("Preparing datasets...")
        
        # Find common samples
        metag_samples = set(self.table_metag_filtered.index)
        metaT_samples = set(self.table_metaT_filtered.index)
        metadata_samples = set(self.metadata_df.index)
        
        self.common_samples = metag_samples.intersection(metaT_samples).intersection(metadata_samples)
        print(f"Common samples: {len(self.common_samples)}")
        
        # Filter to common samples
        self.metag_data = self.table_metag_filtered.loc[self.common_samples]
        self.metaT_data = self.table_metaT_filtered.loc[self.common_samples]
        self.labels = self.metadata_df.loc[self.common_samples, 'diagnosis']
        
        # Encode labels
        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)
        self.label_names = self.label_encoder.classes_
        
        print(f"Label distribution after encoding:")
        for i, label in enumerate(self.label_names):
            count = np.sum(self.encoded_labels == i)
            print(f"  {label}: {count}")
        
        # Scale features
        self.scaler_metag = StandardScaler()
        self.scaler_metaT = StandardScaler()
        
        self.metag_scaled = self.scaler_metag.fit_transform(self.metag_data)
        self.metaT_scaled = self.scaler_metaT.fit_transform(self.metaT_data)
        
        # Create combined dataset for multiomic analysis
        self.combined_data = np.concatenate([self.metag_scaled, self.metaT_scaled], axis=1)
        
        print(f"MetaG scaled shape: {self.metag_scaled.shape}")
        print(f"MetaT scaled shape: {self.metaT_scaled.shape}")
        print(f"Combined shape: {self.combined_data.shape}")

print("MicrobiomeDataProcessor class defined successfully!")


MicrobiomeDataProcessor class defined successfully!



## Cell 5: Define Data Paths and Initialize Processor


In [6]:
# Data paths - update these according to your file locations
data_paths = {
    'metag_biom': 'data/metaG/222424_72996_analysis_Metagenomic_Woltkav014DatabasescratchqpwoltkaWoLr2WoLr2BIOMnonebiom.biom',
    'metat_biom': 'data/metaT/222425_72996_analysis_Metatranscriptomic_Woltkav014DatabasescratchqpwoltkaWoLr2WoLr2BIOMnonebiom.biom',
    'metadata': 'data/metadata/72996_72996_analysis_mapping.txt',
    'lineages': 'data/lineages.txt'
}

# Initialize the data processor
processor = MicrobiomeDataProcessor(data_paths)
print("Data processor initialized successfully!")
print("Ready to load data...")


Data processor initialized successfully!
Ready to load data...



## Cell 6: Load and Process Data


In [7]:

# Load all the data
processor.load_data()



Loading microbiome data...
MetaG features with taxonomy: 10616
MetaT features with taxonomy: 2111
MetaG shape: (1592, 10616)
MetaT shape: (741, 2111)
Metadata shape: (1592, 383)
Diagnosis distribution:
CD        735
UC        444
nonIBD    413
Name: diagnosis, dtype: int64


## Cell 7: Apply RCLR Transformation


In [8]:

# Apply robust CLR transformation
processor.apply_rclr_transformation()


Applying RCLR transformation with pseudocount: 1e-06...
MetaG RCLR shape: (1592, 10616)
MetaT RCLR shape: (741, 2111)



## Cell 8: Filter Features


In [9]:

# Filter features based on prevalence and variance
processor.filter_features(min_prevalence=0.1, top_variance_pct=0.20)

Filtering features (min_prevalence=0.1, top_variance=0.2)...
MetaG features after filtering: 2123
MetaT features after filtering: 422



## Cell 9: Prepare Datasets


In [10]:

# Prepare the final datasets
processor.prepare_datasets()


Preparing datasets...
Common samples: 741
Label distribution after encoding:
  CD: 341
  UC: 212
  nonIBD: 188
MetaG scaled shape: (741, 2123)
MetaT scaled shape: (741, 422)
Combined shape: (741, 2545)


## Cell 10: VAE Model Definition


In [11]:

class MicrobiomeVAE(nn.Module):
    """Enhanced VAE for microbiome data with sparsity handling"""
    
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], latent_dim=50, dropout_rate=0.3):
        super(MicrobiomeVAE, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
            
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent projections
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.LeakyReLU(0.2),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
            
        decoder_layers.append(nn.Linear(hidden_dims[0], input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
        # Input normalization
        self.input_norm = nn.LayerNorm(input_dim)
        
        # Initialize weights properly
        self._initialize_weights()
        
    def _initialize_weights(self):
        """Initialize model weights to prevent NaN losses"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        
    def encode(self, x):
        x = self.input_norm(x)
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_var(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

def microbiome_vae_loss(recon_x, x, mu, logvar, beta=1.0, alpha=0.05):
    """Loss function optimized for microbiome data with stability checks"""
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    
    # KL divergence with clamping to prevent explosion
    logvar = torch.clamp(logvar, -10, 10)  # Clamp logvar to prevent overflow
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    # L1 regularization on latent space for sparsity
    l1_loss = alpha * torch.mean(torch.abs(mu))
    
    total_loss = recon_loss + beta * kl_loss + l1_loss
    
    # Check for NaN and return a large finite value if found
    if torch.isnan(total_loss) or torch.isinf(total_loss):
        print("Warning: NaN or Inf detected in loss, returning fallback loss")
        total_loss = torch.tensor(1e6, device=total_loss.device, requires_grad=True)
        recon_loss = torch.tensor(1e6, device=recon_loss.device)
        kl_loss = torch.tensor(0.0, device=kl_loss.device)
        l1_loss = torch.tensor(0.0, device=l1_loss.device)
    
    return total_loss, recon_loss, kl_loss, l1_loss

print("VAE model and loss function defined successfully!")


VAE model and loss function defined successfully!



## Cell 11: Sparse Autoencoder Model


In [12]:

class SparseAutoencoder(nn.Module):
    """Sparse Autoencoder for feature extraction"""
    
    def __init__(self, latent_dim, sparse_dim, l1_reg=0.001):
        super(SparseAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(latent_dim, sparse_dim),
            nn.ReLU(),
            nn.BatchNorm1d(sparse_dim)
        )
        
        self.decoder = nn.Linear(sparse_dim, latent_dim)
        self.l1_reg = l1_reg
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def get_sparse_features(self, x):
        return self.encoder(x)

def sparse_loss(y_true, y_pred, model, l1_reg=0.001):
    """Loss function for sparse autoencoder"""
    mse_loss = F.mse_loss(y_pred, y_true)
    l1_loss = l1_reg * torch.sum(torch.abs(model.encoder[0].weight))
    return mse_loss + l1_loss

print("Sparse Autoencoder model and loss function defined successfully!")

Sparse Autoencoder model and loss function defined successfully!



## Cell 12: VAE Trainer Class


In [49]:

class MicrobiomeVAETrainer:
    """Trainer class for microbiome VAE"""
    
    def __init__(self, model, device, lr=1e-4):
        self.model = model
        self.device = device
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        
        # Initialize model weights properly
        self._initialize_weights()
        
    def _initialize_weights(self):
        """Initialize model weights properly to avoid NaN losses"""
        for m in self.model.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def train_epoch(self, train_loader, beta=0.01, alpha=0.001, noise_factor=0.01):
        self.model.train()
        total_loss = 0
        valid_batches = 0
        
        for batch_idx, data in enumerate(train_loader):
            data = data[0].to(self.device)
            
            # Skip batches with NaN/inf
            if torch.isnan(data).any() or torch.isinf(data).any():
                continue
            
            # Very small noise to avoid destabilizing training    
            noisy_data = data + torch.randn_like(data) * noise_factor * 0.1
            
            self.optimizer.zero_grad()
            recon_batch, mu, logvar = self.model(noisy_data)
            
            # Skip if model outputs are invalid
            if (torch.isnan(recon_batch).any() or torch.isinf(recon_batch).any() or
                torch.isnan(mu).any() or torch.isinf(mu).any() or
                torch.isnan(logvar).any() or torch.isinf(logvar).any()):
                continue
            
            loss, recon_loss, kl_loss, l1_loss = microbiome_vae_loss(
                recon_batch, data, mu, logvar, beta, alpha
            )
            
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            
            loss.backward()
            
            # Check for valid gradients
            valid_grads = True
            for param in self.model.parameters():
                if param.grad is not None:
                    if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                        valid_grads = False
                        break
            
            if not valid_grads:
                continue
            
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.1)
            self.optimizer.step()
            
            total_loss += loss.item()
            valid_batches += 1
        
        if valid_batches == 0:
            print("Warning: No valid batches in this epoch!")
            return float('inf')
            
        return total_loss / valid_batches
    
    def validate(self, val_loader, beta=0.01, alpha=0.001):
        self.model.eval()
        total_loss = 0
        valid_batches = 0
        
        with torch.no_grad():
            for data in val_loader:
                data = data[0].to(self.device)
                
                if torch.isnan(data).any() or torch.isinf(data).any():
                    continue
                    
                recon_batch, mu, logvar = self.model(data)
                
                if (torch.isnan(recon_batch).any() or torch.isinf(recon_batch).any() or
                    torch.isnan(mu).any() or torch.isinf(mu).any() or
                    torch.isnan(logvar).any() or torch.isinf(logvar).any()):
                    continue
                
                loss, _, _, _ = microbiome_vae_loss(
                    recon_batch, data, mu, logvar, beta, alpha
                )
                
                if not torch.isnan(loss) and not torch.isinf(loss):
                    total_loss += loss.item()
                    valid_batches += 1
        
        if valid_batches == 0:
            return float('inf')
            
        return total_loss / valid_batches
    
    def train(self, train_loader, val_loader, epochs=100, patience=15, 
              beta=0.01, alpha=0.001, noise_factor=0.01):
        """Train the VAE with very conservative parameters"""
        
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        patience_counter = 0
        model_saved = False
        
        print(f"Training VAE for {epochs} epochs with very conservative parameters...")
        start_time = time.time()
        
        for epoch in range(1, epochs + 1):
            train_loss = self.train_epoch(train_loader, beta, alpha, noise_factor)
            val_loss = self.validate(val_loader, beta, alpha)
            
            # Handle infinite losses
            if train_loss == float('inf') or val_loss == float('inf'):
                print(f"Infinite loss at epoch {epoch}, stopping training")
                break
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            if epoch % 10 == 0 or epoch == 1:
                print(f'Epoch {epoch}: Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')
            
            # Early stopping with improvement threshold
            improvement_threshold = 0.0001
            if val_loss < best_val_loss - improvement_threshold:
                best_val_loss = val_loss
                patience_counter = 0
                # Save the best model
                model_path = f'best_vae_model_{self.model.input_dim}.pt'
                torch.save(self.model.state_dict(), model_path)
                model_saved = True
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}')
                break
        
        # Load best model if it was saved
        if model_saved:
            model_path = f'best_vae_model_{self.model.input_dim}.pt'
            if os.path.exists(model_path):
                self.model.load_state_dict(torch.load(model_path))
                print("Loaded best model weights")
            else:
                print("Warning: Best model file not found, using current weights")
        else:
            print("Warning: No model was saved due to training issues")
        
        training_time = time.time() - start_time
        print(f"Training completed in {training_time:.2f} seconds")
        
        return train_losses, val_losses

print("VAE Trainer class defined successfully!")

VAE Trainer class defined successfully!



## Cell 13: Feature Extractor Class


In [89]:

class FeatureExtractor:
    """Extract features using trained VAE and Sparse Autoencoder"""
    
    def __init__(self, vae_model, device):
        self.vae_model = vae_model
        self.device = device
        
    def get_latent_features(self, data):
        """Extract latent features from VAE"""
        self.vae_model.eval()
        
        if isinstance(data, np.ndarray):
            data_tensor = torch.FloatTensor(data).to(self.device)
        else:
            data_tensor = data.to(self.device)
        
        dataset = TensorDataset(data_tensor)
        loader = DataLoader(dataset, batch_size=64)
        
        latent_vectors = []
        with torch.no_grad():
            for batch in loader:
                batch_data = batch[0]
                mu, _ = self.vae_model.encode(batch_data)
                latent_vectors.append(mu.cpu().numpy())
        
        return np.vstack(latent_vectors)
    
    def train_sparse_autoencoder(self, latent_features, sparse_multiplier=3, 
                               epochs=50, l1_reg=0.001):
        """Train sparse autoencoder on latent features with robust NaN handling"""
        
        # Check input features for validity
        if np.isnan(latent_features).any() or np.isinf(latent_features).any():
            print("Warning: Input latent features contain NaN or inf values!")
            # Remove NaN/inf rows
            valid_rows = ~(np.isnan(latent_features).any(axis=1) | np.isinf(latent_features).any(axis=1))
            latent_features = latent_features[valid_rows]
            print(f"Using {valid_rows.sum()} valid samples out of {len(valid_rows)}")
            
            if latent_features.shape[0] < 10:
                print("Error: Too few valid samples for sparse autoencoder training")
                return []
        
        latent_dim = latent_features.shape[1]
        sparse_dim = latent_dim * sparse_multiplier
        
        print(f"Training sparse autoencoder: {latent_dim} -> {sparse_dim}")
        
        sparse_ae = SparseAutoencoder(latent_dim, sparse_dim, l1_reg).to(self.device)
        optimizer = optim.Adam(sparse_ae.parameters(), lr=1e-4)  # Lower learning rate
        
        latent_tensor = torch.FloatTensor(latent_features).to(self.device)
        dataset = TensorDataset(latent_tensor)
        loader = DataLoader(dataset, batch_size=32, shuffle=True)
        
        losses = []
        valid_epochs = 0
        
        for epoch in range(1, epochs + 1):
            sparse_ae.train()
            epoch_loss = 0
            valid_batches = 0
            
            for batch_data in loader:
                data = batch_data[0]
                
                # Skip invalid batches
                if torch.isnan(data).any() or torch.isinf(data).any():
                    continue
                
                optimizer.zero_grad()
                reconstructed = sparse_ae(data)
                
                # Skip if reconstruction is invalid
                if torch.isnan(reconstructed).any() or torch.isinf(reconstructed).any():
                    continue
                
                loss = sparse_loss(data, reconstructed, sparse_ae, l1_reg)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    continue
                
                loss.backward()
                
                # Check gradients
                valid_grads = True
                for param in sparse_ae.parameters():
                    if param.grad is not None:
                        if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                            valid_grads = False
                            break
                
                if not valid_grads:
                    continue
                
                torch.nn.utils.clip_grad_norm_(sparse_ae.parameters(), 1.0)
                optimizer.step()
                
                epoch_loss += loss.item()
                valid_batches += 1
            
            if valid_batches > 0:
                epoch_loss /= valid_batches
                losses.append(epoch_loss)
                valid_epochs += 1
                
                if epoch % 10 == 0 or epoch == 1:
                    print(f'Sparse AE Epoch {epoch}: Loss: {epoch_loss:.6f}')
            else:
                print(f'Sparse AE Epoch {epoch}: No valid batches')
                losses.append(float('inf'))
        
        if valid_epochs == 0:
            print("Error: Sparse autoencoder training failed completely")
            return []
        
        self.sparse_ae = sparse_ae
        print(f"Sparse autoencoder training completed with {valid_epochs} valid epochs")
        return losses
    
    def get_sparse_features(self, latent_features):
        """Extract sparse features with validation"""
        # Check input validity
        if np.isnan(latent_features).any() or np.isinf(latent_features).any():
            print("Warning: Input latent features contain NaN/inf, cleaning...")
            # Replace NaN/inf with zeros or small values
            latent_features = np.nan_to_num(latent_features, nan=0.0, posinf=1.0, neginf=-1.0)
        
        self.sparse_ae.eval()
        latent_tensor = torch.FloatTensor(latent_features).to(self.device)
        
        with torch.no_grad():
            sparse_features = self.sparse_ae.get_sparse_features(latent_tensor)
        
        sparse_array = sparse_features.cpu().numpy()
        
        # Final validation - replace any remaining NaN/inf
        if np.isnan(sparse_array).any() or np.isinf(sparse_array).any():
            print("Warning: Sparse features contain NaN/inf, cleaning...")
            sparse_array = np.nan_to_num(sparse_array, nan=0.0, posinf=1.0, neginf=-1.0)
        
        return sparse_array
    
    def analyze_feature_importance(self, original_feature_names, taxonomy_df=None):
        """Analyze which original features are most important with taxonomy information"""
        
        try:
            # Get weights from sparse autoencoder decoder
            sparse_to_latent = self.sparse_ae.decoder.weight.data.cpu().numpy()
            
            # Get weights from VAE encoder - need to get the right layer
            # The issue is we need to trace back through the encoder properly
            
            # Get the mu projection weights (latent_dim x last_hidden_dim)
            latent_weights = self.vae_model.fc_mu.weight.data.cpu().numpy()
            
            # Get the last encoder layer weights
            encoder_layers = []
            for module in self.vae_model.encoder.modules():
                if isinstance(module, nn.Linear):
                    encoder_layers.append(module)
            
            if len(encoder_layers) == 0:
                print("Warning: Could not find encoder layers for feature importance analysis")
                return {}
                
            # Get the first encoder layer (input -> first hidden)
            first_encoder_weights = encoder_layers[0].weight.data.cpu().numpy()
            
            print(f"Debug shapes:")
            print(f"  Sparse to latent: {sparse_to_latent.shape}")  
            print(f"  Latent weights: {latent_weights.shape}")
            print(f"  First encoder: {first_encoder_weights.shape}")
            print(f"  Original features: {len(original_feature_names)}")
            
            # Check dimension compatibility
            expected_input_dim = first_encoder_weights.shape[1]
            if expected_input_dim != len(original_feature_names):
                print(f"Warning: Dimension mismatch!")
                print(f"  Expected input features: {expected_input_dim}")
                print(f"  Provided feature names: {len(original_feature_names)}")
                
                # Use only the features that match
                original_feature_names = original_feature_names[:expected_input_dim]
                print(f"  Using first {len(original_feature_names)} features")
            
            # For a simplified analysis, just use the first encoder layer weights
            # Each row represents a hidden unit, each column represents an input feature
            feature_importance = {}
            
            # For each sparse feature (decoder output)
            for sparse_idx in range(sparse_to_latent.shape[1]):
                # Get importance as the absolute sum of connections through the network
                # This is simplified - just use first encoder layer importance
                importance_scores = np.abs(first_encoder_weights).mean(axis=0)
                
                # Ensure we don't exceed array bounds
                max_features = min(len(importance_scores), len(original_feature_names))
                importance_scores = importance_scores[:max_features]
                feature_names = original_feature_names[:max_features]
                
                # Create feature importance dictionary with formatted names
                if taxonomy_df is not None:
                    feature_dict = {
                        format_feature_name(feature_name, taxonomy_df): importance_scores[j] 
                        for j, feature_name in enumerate(feature_names)
                    }
                else:
                    feature_dict = dict(zip(feature_names, importance_scores))
                
                feature_importance[sparse_idx] = sorted(feature_dict.items(), 
                                                    key=lambda x: x[1], reverse=True)
            
            return feature_importance
            
        except Exception as e:
            print(f"Error in feature importance analysis: {str(e)}")
            print("Returning empty feature importance dictionary")
            return {}

print("Feature Extractor class defined successfully!")

Feature Extractor class defined successfully!



## Cell 14: Classification Class


In [16]:
class IBDClassifier:
    """Multiclass classifier for IBD prediction"""
    
    def __init__(self):
        self.models = {
            'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
            'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
        }
        
    def train_and_evaluate(self, X_train, X_test, y_train, y_test, label_names):
        """Train multiple models and evaluate performance with feature validation"""
        
        # Validate features before training
        print("Validating features before classification...")
        
        # Check for NaN/inf in training data
        if np.isnan(X_train).any() or np.isinf(X_train).any():
            print("Warning: Training features contain NaN/inf values, cleaning...")
            X_train = np.nan_to_num(X_train, nan=0.0, posinf=1.0, neginf=-1.0)
        
        if np.isnan(X_test).any() or np.isinf(X_test).any():
            print("Warning: Test features contain NaN/inf values, cleaning...")
            X_test = np.nan_to_num(X_test, nan=0.0, posinf=1.0, neginf=-1.0)
        
        # Check feature variance - remove zero variance features
        feature_var = np.var(X_train, axis=0)
        valid_features = feature_var > 1e-10
        
        if not valid_features.all():
            print(f"Removing {(~valid_features).sum()} zero-variance features")
            X_train = X_train[:, valid_features]
            X_test = X_test[:, valid_features]
        
        print(f"Using {X_train.shape[1]} features for classification")
        print(f"Feature range: [{X_train.min():.6f}, {X_train.max():.6f}]")
        
        results = {}
        
        for name, model in self.models.items():
            print(f"\nTraining {name}...")
            
            try:
                # Train model
                model.fit(X_train, y_train)
                
                # Predictions
                y_pred = model.predict(X_test)
                
                # Metrics
                accuracy = accuracy_score(y_test, y_pred)
                f1_micro = f1_score(y_test, y_pred, average='micro')
                f1_macro = f1_score(y_test, y_pred, average='macro')
                f1_weighted = f1_score(y_test, y_pred, average='weighted')
                
                results[name] = {
                    'model': model,
                    'accuracy': accuracy,
                    'f1_micro': f1_micro,
                    'f1_macro': f1_macro,
                    'f1_weighted': f1_weighted,
                    'predictions': y_pred
                }
                
                print(f"Accuracy: {accuracy:.3f}")
                print(f"F1-micro: {f1_micro:.3f}")
                print(f"F1-macro: {f1_macro:.3f}")
                print(f"F1-weighted: {f1_weighted:.3f}")
                
                # Classification report
                print(f"\nClassification Report for {name}:")
                print(classification_report(y_test, y_pred, target_names=label_names))
                
                # Confusion matrix
                self.plot_confusion_matrix(y_test, y_pred, label_names, 
                                         title=f'Confusion Matrix - {name}')
                
            except Exception as e:
                print(f"Error training {name}: {str(e)}")
                results[name] = {
                    'model': None,
                    'accuracy': 0.0,
                    'f1_micro': 0.0,
                    'f1_macro': 0.0,
                    'f1_weighted': 0.0,
                    'predictions': None,
                    'error': str(e)
                }
        
        return results
    
    def plot_confusion_matrix(self, y_true, y_pred, label_names, title='Confusion Matrix'):
        """Plot confusion matrix"""
        cm = confusion_matrix(y_true, y_pred)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=label_names, yticklabels=label_names)
        plt.title(title)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.show()

print("IBD Classifier class defined successfully!")


IBD Classifier class defined successfully!




## Cell 15: Visualization Functions


In [18]:
def visualize_latent_space(latent_vectors, labels, label_names, title="Latent Space"):
    """Visualize latent space with UMAP and t-SNE"""
    
    # UMAP
    print("Computing UMAP embedding...")
    umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    umap_embedding = umap_reducer.fit_transform(latent_vectors)
    
    # t-SNE
    print("Computing t-SNE embedding...")
    tsne_reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_embedding = tsne_reducer.fit_transform(latent_vectors)
    
    # Create visualization DataFrame
    vis_df = pd.DataFrame({
        'UMAP_1': umap_embedding[:, 0],
        'UMAP_2': umap_embedding[:, 1],
        'tSNE_1': tsne_embedding[:, 0],
        'tSNE_2': tsne_embedding[:, 1],
        'Label': [label_names[l] for l in labels]
    })
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # UMAP
    for label in label_names:
        mask = vis_df['Label'] == label
        ax1.scatter(vis_df.loc[mask, 'UMAP_1'], vis_df.loc[mask, 'UMAP_2'], 
                   label=label, alpha=0.7, s=20)
    ax1.set_title(f'UMAP - {title}')
    ax1.set_xlabel('UMAP 1')
    ax1.set_ylabel('UMAP 2')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # t-SNE
    for label in label_names:
        mask = vis_df['Label'] == label
        ax2.scatter(vis_df.loc[mask, 'tSNE_1'], vis_df.loc[mask, 'tSNE_2'], 
                   label=label, alpha=0.7, s=20)
    ax2.set_title(f't-SNE - {title}')
    ax2.set_xlabel('t-SNE 1')
    ax2.set_ylabel('t-SNE 2')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return vis_df

print("Visualization functions defined successfully!")


Visualization functions defined successfully!



## Cell 16: Metagenomics Analysis - Data Preparation


In [20]:
print("="*60)
print("METAGENOMICS ANALYSIS")
print("="*60)

# Prepare data loaders for MetaG
X_train_metag, X_test_metag, y_train, y_test = train_test_split(
    processor.metag_scaled, processor.encoded_labels, 
    test_size=0.2, random_state=42, stratify=processor.encoded_labels
)

print(f"MetaG training set shape: {X_train_metag.shape}")
print(f"MetaG test set shape: {X_test_metag.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

# Create PyTorch tensors and data loaders
train_tensor = torch.FloatTensor(X_train_metag).to(device)
test_tensor = torch.FloatTensor(X_test_metag).to(device)

train_dataset = TensorDataset(train_tensor)
test_dataset = TensorDataset(test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print("Data loaders created successfully!")


METAGENOMICS ANALYSIS
MetaG training set shape: (592, 2123)
MetaG test set shape: (149, 2123)
Training labels shape: (592,)
Test labels shape: (149,)
Data loaders created successfully!



## Cell 17: Metagenomics Analysis - Train VAE


In [22]:
# Create and train VAE for MetaG
input_dim_metag = X_train_metag.shape[1]
print(f"Creating VAE with input dimension: {input_dim_metag}")

vae_metag = MicrobiomeVAE(input_dim_metag, hidden_dims=[512, 256, 128], 
                          latent_dim=50).to(device)

print(f"VAE architecture created:")
print(f"  Input dim: {input_dim_metag}")
print(f"  Hidden dims: [512, 256, 128]")
print(f"  Latent dim: 50")

# Train the VAE
trainer_metag = MicrobiomeVAETrainer(vae_metag, device)
train_losses, val_losses = trainer_metag.train(train_loader, test_loader, epochs=100)

# Plot training curves
plot_training_curves(train_losses, val_losses, "MetaG VAE Training")


Creating VAE with input dimension: 2123
VAE architecture created:
  Input dim: 2123
  Hidden dims: [512, 256, 128]
  Latent dim: 50
Training VAE for 100 epochs with very conservative parameters...
Epoch 1: Train Loss: 0.991817, Val Loss: 1.023663
Epoch 10: Train Loss: 0.992635, Val Loss: 1.044980
Infinite loss at epoch 16, stopping training
Loaded best model weights
Training completed in 74.44 seconds



## Cell 18: Metagenomics Analysis - Extract Latent Features


In [24]:

# Create feature extractor and get latent features
extractor_metag = FeatureExtractor(vae_metag, device)

print("Extracting latent features for all MetaG data...")
all_latent_metag = extractor_metag.get_latent_features(processor.metag_scaled)

print(f"Latent features shape: {all_latent_metag.shape}")
print(f"Latent feature statistics:")
print(f"  Mean: {all_latent_metag.mean():.6f}")
print(f"  Std: {all_latent_metag.std():.6f}")
print(f"  Min: {all_latent_metag.min():.6f}")
print(f"  Max: {all_latent_metag.max():.6f}")


Extracting latent features for all MetaG data...
Latent features shape: (741, 50)
Latent feature statistics:
  Mean: 0.000149
  Std: 0.001356
  Min: -0.005171
  Max: 0.005729



## Cell 19: Metagenomics Analysis - Train Sparse Autoencoder


In [26]:
# Train sparse autoencoder on latent features
print("Training sparse autoencoder for MetaG...")
sparse_losses = extractor_metag.train_sparse_autoencoder(all_latent_metag, 
                                                       sparse_multiplier=3, 
                                                       epochs=50)

# Plot sparse autoencoder training curves
plot_training_curves(sparse_losses, [], "MetaG Sparse Autoencoder Training")


Training sparse autoencoder for MetaG...
Training sparse autoencoder: 50 -> 150
Sparse AE Epoch 1: Loss: 0.526497
Sparse AE Epoch 10: Loss: 0.372199
Sparse AE Epoch 20: Loss: 0.236356
Sparse AE Epoch 30: Loss: 0.132036
Sparse AE Epoch 40: Loss: 0.057928
Sparse AE Epoch 50: Loss: 0.014160
Sparse autoencoder training completed with 50 valid epochs



## Cell 20: Metagenomics Analysis - Extract Sparse Features


In [27]:
# Get sparse features
print("Extracting sparse features...")
sparse_features_metag = extractor_metag.get_sparse_features(all_latent_metag)

print(f"MetaG sparse features shape: {sparse_features_metag.shape}")
print(f"MetaG sparsity: {(sparse_features_metag == 0).mean()*100:.2f}%")
print(f"Non-zero features per sample (mean): {(sparse_features_metag != 0).sum(axis=1).mean():.2f}")
print(f"Feature activation statistics:")
print(f"  Mean: {sparse_features_metag.mean():.6f}")
print(f"  Std: {sparse_features_metag.std():.6f}")
print(f"  Min: {sparse_features_metag.min():.6f}")
print(f"  Max: {sparse_features_metag.max():.6f}")


Extracting sparse features...
MetaG sparse features shape: (741, 150)
MetaG sparsity: 0.00%
Non-zero features per sample (mean): 150.00
Feature activation statistics:
  Mean: 0.000686
  Std: 0.019279
  Min: -0.094996
  Max: 0.069149


## Cell 21: Metagenomics Analysis - Classification


In [28]:
# Split sparse features for classification
X_train_sparse_metag, X_test_sparse_metag = train_test_split(
    sparse_features_metag, test_size=0.2, random_state=42, 
    stratify=processor.encoded_labels
)

print(f"Sparse training features shape: {X_train_sparse_metag.shape}")
print(f"Sparse test features shape: {X_test_sparse_metag.shape}")

# Train and evaluate classifiers
classifier_metag = IBDClassifier()
results_metag = classifier_metag.train_and_evaluate(
    X_train_sparse_metag, X_test_sparse_metag, y_train, y_test, 
    processor.label_names
)

print("MetaG classification completed!")


Sparse training features shape: (592, 150)
Sparse test features shape: (149, 150)
Validating features before classification...
Removing 73 zero-variance features
Using 77 features for classification
Feature range: [-0.094996, 0.069149]

Training LogisticRegression...
Accuracy: 0.456
F1-micro: 0.456
F1-macro: 0.209
F1-weighted: 0.286

Classification Report for LogisticRegression:
              precision    recall  f1-score   support

          CD       0.46      1.00      0.63        68
          UC       0.00      0.00      0.00        43
      nonIBD       0.00      0.00      0.00        38

    accuracy                           0.46       149
   macro avg       0.15      0.33      0.21       149
weighted avg       0.21      0.46      0.29       149


Training RandomForest...
Accuracy: 0.577
F1-micro: 0.577
F1-macro: 0.532
F1-weighted: 0.556

Classification Report for RandomForest:
              precision    recall  f1-score   support

          CD       0.55      0.81      0.65     


## Cell 22: Metagenomics Analysis - Visualization



In [29]:
# Visualize latent space
print("Creating latent space visualizations...")
vis_df_metag = visualize_latent_space(all_latent_metag, processor.encoded_labels, 
                      processor.label_names, "MetaG Latent Space")

print("MetaG analysis completed!")


Creating latent space visualizations...
Computing UMAP embedding...
Computing t-SNE embedding...
MetaG analysis completed!



## Cell 23: Metatranscriptomics Analysis - Data Preparation


In [65]:
print("\n" + "="*60)
print("METATRANSCRIPTOMICS ANALYSIS")
print("="*60)

# Prepare data loaders for MetaT
X_train_metaT, X_test_metaT, y_train, y_test = train_test_split(
    processor.metaT_scaled, processor.encoded_labels, 
    test_size=0.2, random_state=42, stratify=processor.encoded_labels
)

print(f"MetaT training set shape: {X_train_metaT.shape}")
print(f"MetaT test set shape: {X_test_metaT.shape}")

# Create PyTorch tensors and data loaders
train_tensor = torch.FloatTensor(X_train_metaT).to(device)
test_tensor = torch.FloatTensor(X_test_metaT).to(device)

train_dataset = TensorDataset(train_tensor)
test_dataset = TensorDataset(test_tensor)

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=32)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print("MetaT data loaders created successfully!")



METATRANSCRIPTOMICS ANALYSIS
MetaT training set shape: (592, 422)
MetaT test set shape: (149, 422)
MetaT data loaders created successfully!


## Cell 24: Metatranscriptomics Analysis - Train VAE


In [66]:
# Define the simplified VAE class
class SimpleMicrobiomeVAE(nn.Module):
    """Simplified VAE without batch normalization for smaller datasets"""
    
    def __init__(self, input_dim, hidden_dims=[200, 100, 50], latent_dim=25, dropout_rate=0.1):
        super(SimpleMicrobiomeVAE, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder without batch norm
        encoder_layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
            
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent projections
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)
        
        # Decoder without batch norm
        decoder_layers = []
        prev_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = hidden_dim
            
        decoder_layers.append(nn.Linear(hidden_dims[0], input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
        # Conservative weight initialization
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_var(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

print("SimpleMicrobiomeVAE defined successfully!")

SimpleMicrobiomeVAE defined successfully!


In [67]:
print(f"MetaT data stats:")
print(f"  Min: {X_train_metaT.min():.6f}")
print(f"  Max: {X_train_metaT.max():.6f}")
print(f"  Mean: {X_train_metaT.mean():.6f}")
print(f"  Std: {X_train_metaT.std():.6f}")
print(f"  NaN count: {np.isnan(X_train_metaT).sum()}")
print(f"  Inf count: {np.isinf(X_train_metaT).sum()}")

MetaT data stats:
  Min: -3.169552
  Max: 3.168176
  Mean: 0.000029
  Std: 0.996993
  NaN count: 0
  Inf count: 0


In [68]:
# Create and train VAE for MetaT
input_dim_metaT = X_train_metaT.shape[1]
print(f"Creating VAE with input dimension: {input_dim_metaT}")

# vae_metaT = MicrobiomeVAE(input_dim_metaT, hidden_dims=[256, 128, 64], 
#                           latent_dim=32).to(device)
vae_metaT = SimpleMicrobiomeVAE(input_dim_metaT, hidden_dims=[200, 100, 50], 
                               latent_dim=25).to(device)

print(f"VAE architecture created:")
print(f"  Input dim: {input_dim_metaT}")
print(f"  Hidden dims: [256, 128, 64]")
print(f"  Latent dim: 32")

# Train the VAE
trainer_metaT = MicrobiomeVAETrainer(vae_metaT, device)
train_losses, val_losses = trainer_metaT.train(train_loader, test_loader, epochs=100)

# Plot training curves
plot_training_curves(train_losses, val_losses, "MetaT VAE Training")


Creating VAE with input dimension: 422
VAE architecture created:
  Input dim: 422
  Hidden dims: [256, 128, 64]
  Latent dim: 32
Training VAE for 100 epochs with very conservative parameters...
Epoch 1: Train Loss: 0.994022, Val Loss: 1.034346
Epoch 10: Train Loss: 0.993923, Val Loss: 1.034718
Early stopping at epoch 16
Loaded best model weights
Training completed in 223.19 seconds



## Cell 25: Metatranscriptomics Analysis - Extract Features and Train SAE



In [69]:
# Create feature extractor and get latent features
extractor_metaT = FeatureExtractor(vae_metaT, device)

print("Extracting latent features for all MetaT data...")
all_latent_metaT = extractor_metaT.get_latent_features(processor.metaT_scaled)

print(f"Latent features shape: {all_latent_metaT.shape}")

# Train sparse autoencoder
print("Training sparse autoencoder for MetaT...")
sparse_losses = extractor_metaT.train_sparse_autoencoder(all_latent_metaT, 
                                                       sparse_multiplier=3, 
                                                       epochs=50)

plot_training_curves(sparse_losses, [], "MetaT Sparse Autoencoder Training")

# Get sparse features
sparse_features_metaT = extractor_metaT.get_sparse_features(all_latent_metaT)


Extracting latent features for all MetaT data...
Latent features shape: (741, 25)
Training sparse autoencoder for MetaT...
Training sparse autoencoder: 25 -> 75
Sparse AE Epoch 1: Loss: 0.192288
Sparse AE Epoch 10: Loss: 0.149393
Sparse AE Epoch 20: Loss: 0.111835
Sparse AE Epoch 30: Loss: 0.079760
Sparse AE Epoch 40: Loss: 0.053202
Sparse AE Epoch 50: Loss: 0.032172
Sparse autoencoder training completed with 50 valid epochs


In [70]:

print(f"MetaT sparse features shape: {sparse_features_metaT.shape}")
print(f"MetaT sparsity: {(sparse_features_metaT == 0).mean()*100:.2f}%")


MetaT sparse features shape: (741, 75)
MetaT sparsity: 0.00%



## Cell 26: Metatranscriptomics Analysis - Classification and Visualization


In [71]:
# Split sparse features for classification
X_train_sparse_metaT, X_test_sparse_metaT = train_test_split(
    sparse_features_metaT, test_size=0.2, random_state=42, 
    stratify=processor.encoded_labels
)

# Train and evaluate classifiers
classifier_metaT = IBDClassifier()
results_metaT = classifier_metaT.train_and_evaluate(
    X_train_sparse_metaT, X_test_sparse_metaT, y_train, y_test, 
    processor.label_names
)

# Visualize latent space
vis_df_metaT = visualize_latent_space(all_latent_metaT, processor.encoded_labels, 
                      processor.label_names, "MetaT Latent Space")

print("MetaT analysis completed!")



Validating features before classification...
Removing 37 zero-variance features
Using 38 features for classification
Feature range: [-0.038262, 0.037389]

Training LogisticRegression...
Accuracy: 0.456
F1-micro: 0.456
F1-macro: 0.209
F1-weighted: 0.286

Classification Report for LogisticRegression:
              precision    recall  f1-score   support

          CD       0.46      1.00      0.63        68
          UC       0.00      0.00      0.00        43
      nonIBD       0.00      0.00      0.00        38

    accuracy                           0.46       149
   macro avg       0.15      0.33      0.21       149
weighted avg       0.21      0.46      0.29       149


Training RandomForest...
Accuracy: 0.510
F1-micro: 0.510
F1-macro: 0.419
F1-weighted: 0.460

Classification Report for RandomForest:
              precision    recall  f1-score   support

          CD       0.52      0.85      0.64        68
          UC       0.45      0.21      0.29        43
      nonIBD       0.5

## Cell 29: Results Summary and Comparison (Modified)


In [77]:

print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)

# Compare results across approaches (only MetaG and MetaT)
approaches = ['MetaG', 'MetaT']
all_results = [results_metag, results_metaT]

summary_data = []

for approach, results in zip(approaches, all_results):
    for model_name, metrics in results.items():
        if 'error' not in metrics:  # Only include successful results
            summary_data.append({
                'Approach': approach,
                'Model': model_name,
                'Accuracy': metrics['accuracy'],
                'F1_Micro': metrics['f1_micro'],
                'F1_Macro': metrics['f1_macro'],
                'F1_Weighted': metrics['f1_weighted']
            })

summary_df = pd.DataFrame(summary_data)
print("Performance Summary:")
print(summary_df.to_string(index=False, float_format='%.3f'))

# Find best performing model
if len(summary_df) > 0:
    best_idx = summary_df['F1_Weighted'].idxmax()
    best_result = summary_df.iloc[best_idx]
    print(f"\nBest performing model:")
    print(f"  Approach: {best_result['Approach']}")
    print(f"  Model: {best_result['Model']}")
    print(f"  F1-Weighted: {best_result['F1_Weighted']:.3f}")
    
    # Create performance comparison plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    metrics = ['Accuracy', 'F1_Micro', 'F1_Macro', 'F1_Weighted']
    
    for i, metric in enumerate(metrics):
        ax = axes[i//2, i%2]
        
        # Group by approach
        for approach in approaches:
            approach_data = summary_df[summary_df['Approach'] == approach]
            if len(approach_data) > 0:
                ax.bar([f"{approach}_{row['Model']}" for _, row in approach_data.iterrows()], 
                      approach_data[metric], alpha=0.7, label=approach)
        
        ax.set_title(f'{metric} Comparison')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No successful results to summarize.")




RESULTS SUMMARY
Performance Summary:
Approach              Model  Accuracy  F1_Micro  F1_Macro  F1_Weighted
   MetaG LogisticRegression     0.456     0.456     0.209        0.286
   MetaG       RandomForest     0.577     0.577     0.532        0.556
   MetaT LogisticRegression     0.456     0.456     0.209        0.286
   MetaT       RandomForest     0.510     0.510     0.419        0.460

Best performing model:
  Approach: MetaG
  Model: RandomForest
  F1-Weighted: 0.556



## Cell 30: Feature Importance Analysis (Modified)


In [91]:

print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

if len(summary_df) > 0:
    # Get feature importance based on the best approach
    best_approach = best_result['Approach']
    
    print(f"Analyzing feature importance for best approach: {best_approach}")
    
    if best_approach == 'MetaG':
        feature_importance = extractor_metag.analyze_feature_importance(
            processor.metag_data.columns, 
            processor.metag_taxonomy.set_index('genome_id') if hasattr(processor, 'metag_taxonomy') else None
        )
        sparse_activations = sparse_features_metag.mean(axis=0)
        sparse_data = sparse_features_metag
        print(f"Top important MetaG features:")
        
    elif best_approach == 'MetaT':
        feature_importance = extractor_metaT.analyze_feature_importance(
            processor.metaT_data.columns, 
            processor.metaT_taxonomy.set_index('genome_id') if hasattr(processor, 'metaT_taxonomy') else None
        )
        sparse_activations = sparse_features_metaT.mean(axis=0)
        sparse_data = sparse_features_metaT
        print(f"Top important MetaT features:")
    
    # Show top sparse features and their important genes
    top_sparse_features = np.argsort(sparse_activations)[-5:]
    
    for i, sparse_idx in enumerate(top_sparse_features):
        print(f"\nSparse feature {sparse_idx} (activation: {sparse_activations[sparse_idx]:.4f}):")
        if sparse_idx in feature_importance:
            top_genes = feature_importance[sparse_idx][:10]
            for gene, importance in top_genes:
                print(f"  {gene}: {importance:.4f}")
        else:
            print("  No feature importance data available")
    
    # Create feature activation heatmap for top sparse features
    if len(top_sparse_features) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Create heatmap of top sparse features across samples
        top_features_data = sparse_data[:, top_sparse_features].T
        
        sns.heatmap(top_features_data, 
                   yticklabels=[f'Sparse_{i}' for i in top_sparse_features],
                   cmap='viridis', ax=ax)
        ax.set_title(f'Top 5 Sparse Features Activation - {best_approach}')
        ax.set_xlabel('Samples')
        ax.set_ylabel('Sparse Features')
        plt.tight_layout()
        plt.show()
else:
    print("No results available for feature importance analysis.")




FEATURE IMPORTANCE ANALYSIS
Analyzing feature importance for best approach: MetaG


IndexError: index 128 is out of bounds for axis 0 with size 128

In [92]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

if len(summary_df) > 0:
    # Get feature importance based on the best approach
    best_approach = best_result['Approach']
    
    print(f"Analyzing feature importance for best approach: {best_approach}")
    
    if best_approach == 'MetaG':
        sparse_ae = extractor_metag.sparse_ae
        vae_model = vae_metag
        gene_names = processor.metag_data.columns
        sparse_features = sparse_features_metag
        sparse_dim = sparse_features_metag.shape[1]
        
    elif best_approach == 'MetaT':
        sparse_ae = extractor_metaT.sparse_ae
        vae_model = vae_metaT
        gene_names = processor.metaT_data.columns
        sparse_features = sparse_features_metaT
        sparse_dim = sparse_features_metaT.shape[1]
    
    # Get the weights
    print("Extracting weights from models...")
    vae_encoder_weights = vae_model.fc_mu.weight.data.cpu().numpy()
    
    print(f"Shapes:")
    print(f"  VAE encoder weights: {vae_encoder_weights.shape}")
    print(f"  Sparse AE decoder: {sparse_ae.decoder.weight.shape}")
    print(f"  Gene names: {len(gene_names)}")
    
    # Analyze sparse feature importance
    print("Analyzing sparse feature importance...")
    feature_importance = {}
    for i in range(sparse_dim):
        # Get weights for this sparse feature
        sparse_to_latent = sparse_ae.decoder.weight[:, i].cpu().detach().numpy()
        
        # Multiply with VAE encoder weights to get effect on original genes
        # maps: sparse feature -> latent space -> original genes
        gene_importance = np.abs(np.dot(vae_encoder_weights.T, sparse_to_latent))
        
        # Map to gene names
        feature_importance[i] = dict(zip(gene_names, gene_importance))
    
    # Find top genes for each sparse feature
    top_genes_per_feature = {}
    for feature_idx, gene_scores in feature_importance.items():
        # Sort genes by importance score
        sorted_genes = sorted(gene_scores.items(), key=lambda x: x[1], reverse=True)
        # Keep top 10 genes
        top_genes_per_feature[feature_idx] = sorted_genes[:10]
    
    # Get feature activations
    feature_activations = sparse_features.mean(axis=0)
    top_activated_features = np.argsort(feature_activations)[-5:]
    
    print(f"\nTop genes for most activated sparse features in {best_approach}:")
    for i, feature_idx in enumerate(top_activated_features):
        print(f"\nSparse feature {feature_idx} (activation: {feature_activations[feature_idx]:.4f}):")
        for gene, score in top_genes_per_feature[feature_idx]:
            print(f"  {gene}: {score:.4f}")
    
    # Calculate overall gene importance
    gene_overall_importance = {gene: 0 for gene in gene_names}
    for feature_idx, gene_scores in feature_importance.items():
        activation = feature_activations[feature_idx]
        for gene, score in gene_scores.items():
            gene_overall_importance[gene] += score * activation
    
    # Create importance dataframe
    sparse_importance_df = pd.DataFrame({
        'Gene': list(gene_overall_importance.keys()),
        'Sparse_AE_Importance': list(gene_overall_importance.values())
    })
    sparse_importance_df = sparse_importance_df.sort_values('Sparse_AE_Importance', ascending=False)
    
    print(f"\nTop 20 genes by Sparse Autoencoder importance in {best_approach}:")
    print(sparse_importance_df.head(20))
    
    # Save results
    sparse_importance_df.to_csv(f'sparse_importance_{best_approach.lower()}.csv', index=False)
    print(f"Results saved to sparse_importance_{best_approach.lower()}.csv")
    
    # Create feature activation heatmap for top sparse features
    if len(top_activated_features) > 0:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Create heatmap of top sparse features across samples
        top_features_data = sparse_features[:, top_activated_features].T
        
        sns.heatmap(top_features_data, 
                   yticklabels=[f'Sparse_{i}' for i in top_activated_features],
                   cmap='viridis', ax=ax)
        ax.set_title(f'Top 5 Sparse Features Activation - {best_approach}')
        ax.set_xlabel('Samples')
        ax.set_ylabel('Sparse Features')
        plt.tight_layout()
        plt.show()
        
else:
    print("No results available for feature importance analysis.")


FEATURE IMPORTANCE ANALYSIS
Analyzing feature importance for best approach: MetaG
Extracting weights from models...
Shapes:
  VAE encoder weights: (50, 128)
  Sparse AE decoder: torch.Size([50, 150])
  Gene names: 2123
Analyzing sparse feature importance...

Top genes for most activated sparse features in MetaG:

Sparse feature 14 (activation: 0.0239):
  G000178255: 0.0084
  G001298465: 0.0084
  G009937725: 0.0081
  G000437075: 0.0079
  G001647615: 0.0069
  G004522215: 0.0065
  G000158275: 0.0062
  G003243265: 0.0059
  G000159435: 0.0055
  G001578645: 0.0054

Sparse feature 71 (activation: 0.0242):
  G902770605: 0.0122
  G000424605: 0.0108
  G000759795: 0.0096
  G000324145: 0.0084
  G000437915: 0.0075
  G004961775: 0.0074
  G000168835: 0.0074
  G900549925: 0.0072
  G001917135: 0.0068
  G000437075: 0.0067

Sparse feature 67 (activation: 0.0245):
  G001602485: 0.0089
  G003343565: 0.0085
  G000437175: 0.0079
  G007581405: 0.0076
  G000437915: 0.0073
  G000262305: 0.0072
  G000980515: 0.


## Cell 31: Data Quality and Model Diagnostics (Modified)


In [93]:

print("\n" + "="*60)
print("DATA QUALITY AND MODEL DIAGNOSTICS")
print("="*60)

# Data quality summary
print("Data Quality Summary:")
print(f"Total samples: {len(processor.common_samples)}")
print(f"MetaG features (filtered): {processor.metag_scaled.shape[1]}")
print(f"MetaT features (filtered): {processor.metaT_scaled.shape[1]}")

# Class distribution
print(f"\nClass distribution:")
for i, label in enumerate(processor.label_names):
    count = np.sum(processor.encoded_labels == i)
    print(f"  {label}: {count} ({count/len(processor.encoded_labels)*100:.1f}%)")

# Model diagnostics
print(f"\nModel Architecture Summary:")
print(f"VAE Latent Dimensions:")
print(f"  MetaG: {vae_metag.latent_dim}")
print(f"  MetaT: {vae_metaT.latent_dim}")

# Feature sparsity analysis
sparsity_data = []
if 'sparse_features_metag' in locals():
    sparsity_data.append(['MetaG', (sparse_features_metag == 0).mean() * 100])
if 'sparse_features_metaT' in locals():
    sparsity_data.append(['MetaT', (sparse_features_metaT == 0).mean() * 100])

if sparsity_data:
    sparsity_df = pd.DataFrame(sparsity_data, columns=['Approach', 'Sparsity (%)'])
    print(f"\nSparse Feature Statistics:")
    print(sparsity_df.to_string(index=False, float_format='%.2f'))

# Training convergence check
print(f"\nTraining Convergence Analysis:")
if len(train_losses) > 10:
    final_losses = train_losses[-10:]
    loss_std = np.std(final_losses)
    print(f"  Final 10 epochs loss std: {loss_std:.6f}")
    if loss_std < 0.01:
        print("  ✓ Training appears converged")
    else:
        print("  ⚠ Training may not have fully converged")
else:
    print("  Insufficient training epochs for convergence analysis")




DATA QUALITY AND MODEL DIAGNOSTICS
Data Quality Summary:
Total samples: 741
MetaG features (filtered): 2123
MetaT features (filtered): 422

Class distribution:
  CD: 341 (46.0%)
  UC: 212 (28.6%)
  nonIBD: 188 (25.4%)

Model Architecture Summary:
VAE Latent Dimensions:
  MetaG: 50
  MetaT: 25

Sparse Feature Statistics:
Approach  Sparsity (%)
   MetaG          0.00
   MetaT          0.00

Training Convergence Analysis:
  Final 10 epochs loss std: 0.000976
  ✓ Training appears converged



## Cell 32: Save Results and Models (Modified)


In [81]:

print("\n" + "="*60)
print("SAVING RESULTS AND MODELS")
print("="*60)

# Create results dictionary (without combined results)
pipeline_results = {
    'processor': processor,
    'results_metag': results_metag,
    'results_metaT': results_metaT,
    'summary': summary_df,
    'models': {
        'vae_metag': vae_metag,
        'vae_metaT': vae_metaT
    },
    'extractors': {
        'extractor_metag': extractor_metag,
        'extractor_metaT': extractor_metaT
    }
}

# Save model states
print("Saving model states...")
torch.save(vae_metag.state_dict(), 'vae_metag_final.pt')
torch.save(vae_metaT.state_dict(), 'vae_metaT_final.pt')

# Save sparse features
print("Saving sparse features...")
np.save('sparse_features_metag.npy', sparse_features_metag)
np.save('sparse_features_metaT.npy', sparse_features_metaT)

# Save results summary
summary_df.to_csv('results_summary.csv', index=False)

# Save latent representations
print("Saving latent representations...")
np.save('latent_metag.npy', all_latent_metag)
np.save('latent_metaT.npy', all_latent_metaT)

print("All results and models saved successfully!")





SAVING RESULTS AND MODELS
Saving model states...
Saving sparse features...
Saving latent representations...
All results and models saved successfully!


## Cell 33: Final Summary and Conclusions (Modified)


In [82]:

print("\n" + "="*80)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("="*80)

print("Summary of Completed Analyses:")
print("✓ Data loading and preprocessing")
print("✓ RCLR transformation applied")
print("✓ Feature filtering and selection")
print("✓ Individual MetaG analysis with VAE+SAE")
print("✓ Individual MetaT analysis with VAE+SAE")
print("✓ Classification performance evaluation")
print("✓ Feature importance analysis")
print("✓ Latent space visualization")
print("✓ Results saved to files")

if len(summary_df) > 0:
    print(f"\nBest Performance Achieved:")
    print(f"  Method: {best_result['Approach']} with {best_result['Model']}")
    print(f"  Accuracy: {best_result['Accuracy']:.3f}")
    print(f"  F1-Weighted: {best_result['F1_Weighted']:.3f}")

print(f"\nOutput Files Generated:")
print("  - vae_metag_final.pt (MetaG VAE model)")
print("  - vae_metaT_final.pt (MetaT VAE model)")
print("  - sparse_features_*.npy (Sparse feature arrays)")
print("  - latent_*.npy (Latent representation arrays)")
print("  - results_summary.csv (Performance summary)")

print(f"\nNext Steps:")
print("  1. Analyze feature importance results for biological insights")
print("  2. Validate findings on independent test set")
print("  3. Investigate top discriminative microbial features")
print("  4. Consider ensemble methods for improved performance")
print("  5. Explore temporal dynamics if longitudinal data available")
print("  6. Consider multiomic integration approaches if needed")

print(f"\nPipeline variables available in workspace:")
print("  - processor: Data processing object")
print("  - vae_metag, vae_metaT: Trained VAE models")
print("  - extractor_*: Feature extraction objects")
print("  - results_*: Classification results dictionaries")
print("  - sparse_features_*: Extracted sparse features")
print("  - all_latent_*: Latent representations")
print("  - summary_df: Performance comparison dataframe")

print("\n" + "="*80)


PIPELINE COMPLETED SUCCESSFULLY
Summary of Completed Analyses:
✓ Data loading and preprocessing
✓ RCLR transformation applied
✓ Feature filtering and selection
✓ Individual MetaG analysis with VAE+SAE
✓ Individual MetaT analysis with VAE+SAE
✓ Classification performance evaluation
✓ Feature importance analysis
✓ Latent space visualization
✓ Results saved to files

Best Performance Achieved:
  Method: MetaG with RandomForest
  Accuracy: 0.577
  F1-Weighted: 0.556

Output Files Generated:
  - vae_metag_final.pt (MetaG VAE model)
  - vae_metaT_final.pt (MetaT VAE model)
  - sparse_features_*.npy (Sparse feature arrays)
  - latent_*.npy (Latent representation arrays)
  - results_summary.csv (Performance summary)

Next Steps:
  1. Analyze feature importance results for biological insights
  2. Validate findings on independent test set
  3. Investigate top discriminative microbial features
  4. Consider ensemble methods for improved performance
  5. Explore temporal dynamics if longitudinal